# From French balancing data to a simulated BESS response

> **Demonstration question:** If a hypothetical battery accepts fixed aFRR or mFRR commitments, can Open Energy Lab replay a clearly derived activation request, enforce the battery limits, and explain the resulting delivery and settlement?

This is an **offline, auditable worked example**. It reproduces two Feature 011 reference studies from governed artifacts; it is not observed battery dispatch and not a forecast, bid, optimization, or profitability guarantee.

### The evidence chain

```text
OBSERVED MARKET DATA          EXPLICIT STUDY ASSUMPTIONS       OPEN ENERGY LAB             INTERPRETABLE OUTPUTS
RTE aFRR/mFRR aggregates  ->  0.001 allocation + fixed bids ->  deterministic BESS replay -> delivery, SOC, headroom
RTE area price proxies        zero baseline + battery limits    validation + settlement     performance, cost, revenue
ENTSO-E A44 (optional)        no hidden interpolation            reproducible fingerprints   assumptions and limitations
```

Throughout the notebook, headings and plot titles identify the evidence level: **OBSERVED** comes from a governed market artifact, **ASSUMED / DERIVED** constructs a research scenario, and **SIMULATED** is produced by Open Energy Lab.

> **Restricted-data notice:** The RTE-derived files are private-repository data for authorized collaborators. Do not publish the executed notebook, screenshots, HTML exports, figures, or public CI artifacts. The source notebook is committed with empty outputs; remove or reassess the restricted fixtures before a visibility change.


## 1. What aFRR and mFRR mean in this demonstration

**Automatic Frequency Restoration Reserve (aFRR)** helps bring the power system back toward balance automatically after the fastest frequency response has acted. In an operating system, a participating asset would normally follow a much faster automatic control signal. Here, RTE provides only 15-minute area summaries, so the example tests **interval-average energy and physical feasibility**, not second-by-second control quality.

**Manual Frequency Restoration Reserve (mFRR)** is activated as a more deliberate balancing action. **Scheduled activation (SA)** represents a planned quarter-hour activation in the balancing process. **Direct activation (DA)** can be called directly when needed. The selected public record has source-null DA, so no direct event is fabricated. The SA replay uses a declared rectangular 15-minute event; that shape is a simulation assumption, not an observed asset instruction.

The two cases remain separate: aFRR is replayed as an automatic interval-average trajectory, while mFRR is replayed as a scheduled event with explicit timing and duration. They are not interchangeable with each other or with FCR.


## 2. Data roles and evidence boundaries

| Layer | Provider / owner | Role in this notebook | What it does **not** establish |
|---|---|---|---|
| **OBSERVED** | RTE Balancing Energy API 5.2 | French control-area aFRR and mFRR volumes and associated area prices | A command addressed to the reference battery |
| **OPTIONAL CONTEXT** | ENTSO-E Transparency Platform | A time-compatible A44 `day_ahead_price` may be selected from the existing Feature 010 profile | aFRR/mFRR activation, an automatic fallback, or a value used by these offline reference settlements |
| **ASSUMED / DERIVED** | This study | Fixed allocation, commitment, baseline, event shape, battery and economic assumptions | An observed bid, award, control signal, contract, or invoice |
| **SIMULATED** | Open Energy Lab | Battery dispatch, SOC/headroom, delivery validation, throughput and settlement arithmetic | A forecast or operational recommendation |

RTE Balancing Energy 5.2 publishes French control-area aggregates, not commands sent to an individual battery. The reference scenarios apply one fixed 0.001 allocation and label every result `derived_simulation`. The aFRR inputs remain 15-minute interval averages with no interpolation or upsampling. The mFRR event shape is an explicit rectangular simulation assumption.

The audit keeps source field / series fingerprint / unit / resolution visible. For aFRR it separates requested need / activated for France / activated in France / weighted-average price. For mFRR it separates SA requested / SA satisfied / SA activated / DA status / clearing price. Prices are labelled `scenario_proxy`, not an asset invoice. Capacity price, day-ahead price, and penalties are zero or unconfigured in these fixed offline studies; therefore the settlement plot is deliberately not a complete commercial business case. The degradation value is a throughput comparison proxy, not an electrochemical aging model.


## 3. Load the exact offline evidence

The notebook first reloads the normalized RTE artifacts, their derived scenario definitions, and the recorded expected results. It rebuilds both replays and requires an exact match before displaying anything. This makes the walkthrough repeatable without contacting RTE or ENTSO-E and prevents a changed input from quietly producing a different story.

The provenance table answers **where did this case come from?** The fingerprints link each visible result back through source artifact, mapping, activation, replay, and settlement. They are audit identities—not market indicators.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").is_file() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("run the notebook from within the Open Energy Lab checkout")
FIXTURE_DIR = REPO_ROOT / "tests/fixtures/restoration"
sns.set_theme(style="whitegrid", context="notebook")
generated_figures = {}


In [ ]:
from examples.restoration_services.generate_reference_results import (
    AFRR_SCENARIO_FILENAME,
    EXPECTED_RESULTS_FILENAME,
    MFRR_SCENARIO_FILENAME,
    build_reference_case,
    build_reference_display_payload,
    build_reference_results_payload,
    load_reference_scenario,
)

afrr_scenario = load_reference_scenario(FIXTURE_DIR / "france_2026-08-20_afrr_scenario_v1.json", fixture_directory=FIXTURE_DIR)
mfrr_scenario = load_reference_scenario(FIXTURE_DIR / "france_2026-08-15_mfrr_scenario_v1.json", fixture_directory=FIXTURE_DIR)
product_cases = {
    "AFRR": build_reference_case(afrr_scenario),
    "MFRR": build_reference_case(mfrr_scenario),
}
rebuilt_results = build_reference_results_payload(FIXTURE_DIR)
recorded_results = json.loads((FIXTURE_DIR / "france_restoration_expected_results_v1.json").read_text(encoding="utf-8"))
assert rebuilt_results == recorded_results
display_payload = build_reference_display_payload(FIXTURE_DIR)
result_set_fingerprint = rebuilt_results["result_set_fingerprint"]
notebook_display_fingerprint = display_payload["notebook_display_fingerprint"]
print(f"result_set_fingerprint={result_set_fingerprint}")
print(f"notebook_display_fingerprint={notebook_display_fingerprint}")


In [ ]:
provenance_table = pd.DataFrame(display_payload["provenance_rows"])
display(provenance_table[["product", "source", "source_version", "resource", "requested_start_utc", "requested_end_utc", "output_label", "redistribution"]])

fig, axis = plt.subplots(figsize=(13, 3.2))
axis.set_xlim(-0.5, 3.5)
axis.set_ylim(-0.8, 0.8)
axis.axis("off")

stages = [
    ("OBSERVED", "RTE 5.2 area data", "#4C78A8"),
    ("ASSUMED / DERIVED", "0.001 allocation\nfixed commitments and baseline", "#F2CF5B"),
    ("SIMULATED", "Open Energy Lab\nbattery replay", "#54A24B"),
    ("INTERPRETED", "delivery, headroom\nand settlement", "#B279A2"),
]
for index, (level, description, color) in enumerate(stages):
    axis.text(index, 0, f"{level}\n{description}", ha="center", va="center", fontsize=10, bbox={"boxstyle": "round,pad=0.6", "facecolor": color, "alpha": 0.22, "edgecolor": color})
    if index < len(stages) - 1:
        axis.annotate("", xy=(index + 0.63, 0), xytext=(index + 0.37, 0), arrowprops={"arrowstyle": "->", "lw": 2, "color": "#555555"})
axis.set_title("How evidence becomes a simulation result", fontsize=14, pad=14)
fig.tight_layout()
generated_figures["evidence_chain"] = fig
plt.show()


### 3.1 What is actually observed

The series catalogue below is the source-of-truth inventory. `identity=observed_area_aggregate` means the value was preserved from the RTE market-area response. Only one explicitly named **driver series** is later mapped into each battery scenario; the remaining series stay as comparison context and are never silently substituted.

- aFRR driver: upward requested need. Activated volumes and weighted-average prices are comparisons or settlement proxies.
- mFRR driver: downward SA requested volume. Satisfied and activated volumes describe the area outcome; the DA fields remain missing when RTE returned null.

A missing source value is not zero. The normalized `value_state` preserves that distinction.


In [ ]:
afrr_market_table = pd.DataFrame(display_payload["afrr_market_rows"])
mfrr_market_table = pd.DataFrame(display_payload["mfrr_market_rows"])
market_series_catalog = pd.concat([afrr_market_table, mfrr_market_table], ignore_index=True).groupby(["product", "series_id"], as_index=False).agg(source_field=("source_field", "first"), direction=("direction", "first"), unit=("unit", "first"), resolution=("resolution", "first"), identity=("identity", "first"), observations=("start_utc", "size"))
display(market_series_catalog)
display(afrr_market_table.pivot(index="start_utc", columns="series_id", values="value"))
display(mfrr_market_table[["start_utc", "series_id", "direction", "value_state", "value", "unit"]])


### 3.2 aFRR case — read the market before the battery

**OBSERVED:** four PT15M records cover one hour. The upper panel compares requested need with RTE's two activation aggregates; the lower panel shows the associated weighted-average area prices. These are market-area magnitudes. The battery sign convention and allocation are introduced only in the next stage.

**What to look for:** the requested series varies across the four quarters. It is sufficiently resolved for this interval-average demonstration, but it cannot reveal the faster automatic regulation signal within each quarter-hour.


In [ ]:
afrr_plot = afrr_market_table.copy()
afrr_plot["start_utc"] = pd.to_datetime(afrr_plot["start_utc"], utc=True)
afrr_labels = {
    "afrr_requested_need_upward": "Requested need — upward (driver)",
    "afrr_requested_need_downward": "Requested need — downward",
    "afrr_activated_for_fr_upward": "Activated for France — upward",
    "afrr_activated_for_fr_downward": "Activated for France — downward",
    "afrr_activated_in_fr_upward": "Activated in France — upward",
    "afrr_activated_in_fr_downward": "Activated in France — downward",
    "afrr_weighted_average_price_upward": "Weighted-average price — upward",
    "afrr_weighted_average_price_downward": "Weighted-average price — downward",
}
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for series_id, group in afrr_plot[afrr_plot["unit"] == "MW"].groupby("series_id"):
    is_driver = series_id == "afrr_requested_need_upward"
    axes[0].plot(group["start_utc"], group["value"], marker="o", linewidth=2.8 if is_driver else 1.5, linestyle="-" if is_driver else "--", label=afrr_labels[series_id])
axes[0].set_ylabel("French area volume (MW)")
axes[0].set_title("OBSERVED — RTE aFRR area volumes (driver emphasized)")
axes[0].legend(fontsize=8, ncol=2)
for series_id, group in afrr_plot[afrr_plot["unit"] == "EUR/MWh"].groupby("series_id"):
    axes[1].plot(group["start_utc"], group["value"], marker="o", label=afrr_labels[series_id])
axes[1].set_ylabel("Area price (EUR/MWh)")
axes[1].set_xlabel("UTC interval start")
axes[1].set_title("OBSERVED — RTE weighted-average activation-price context")
axes[1].legend(fontsize=8)
fig.tight_layout()
generated_figures["afrr_market"] = fig
plt.show()


### 3.3 mFRR case — a single scheduled quarter-hour

**OBSERVED:** this selected PT15M record contains a downward scheduled-activation request and satisfied volume. It is intentionally a bounded example, so bars communicate the one-quarter snapshot better than a time-series line. The direct-activation value is source-null and is marked **SOURCE NULL**, not plotted as a measured zero.

**What to look for:** requested, satisfied and activated are different market concepts. Only requested SA is used as the driver of the derived event. The clearing price supplies scenario-price context; it is not proof of the battery's contracted settlement.


In [ ]:
mfrr_plot = mfrr_market_table.copy()
mfrr_plot["start_utc"] = pd.to_datetime(mfrr_plot["start_utc"], utc=True)
mfrr_labels = {
    "mfrr_sa_requested_volume_downward": "SA requested (driver)",
    "mfrr_sa_satisfied_need_downward": "SA satisfied need",
    "mfrr_sa_activated_volume_downward": "SA activated in France",
    "mfrr_da_requested_volume_downward": "DA requested",
}
selected_mfrr_ids = list(mfrr_labels)
volume_rows = mfrr_plot[mfrr_plot["series_id"].isin(selected_mfrr_ids)].drop_duplicates("series_id").set_index("series_id").reindex(selected_mfrr_ids).reset_index()
fig, axes = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={"height_ratios": [2, 1]})
bar_values = volume_rows["value"].fillna(0.0)
bars = axes[0].bar([mfrr_labels[item] for item in volume_rows["series_id"]], bar_values, color=["#F58518", "#4C78A8", "#54A24B", "#B8B8B8"])
for bar, row in zip(bars, volume_rows.itertuples(), strict=True):
    if row.value_state != "VALUE":
        bar.set_hatch("//")
        axes[0].annotate("SOURCE NULL\n(not zero)", (bar.get_x() + bar.get_width() / 2, 0), xytext=(0, 8), textcoords="offset points", ha="center", va="bottom", fontsize=9)
axes[0].set_ylabel("French area volume (MW)")
axes[0].set_title("OBSERVED — RTE mFRR scheduled context and sparse direct activation")
axes[0].tick_params(axis="x", rotation=12)
price_rows = mfrr_plot[mfrr_plot["series_id"] == "mfrr_clearing_price_downward"]
axes[1].bar(["Downward clearing price"], [price_rows["value"].iloc[0]], color="#E45756")
axes[1].set_ylabel("Area price (EUR/MWh)")
axes[1].set_title("OBSERVED — RTE clearing-price context")
fig.tight_layout()
generated_figures["mfrr_market"] = fig
plt.show()


## 4. Construct an explicit research scenario

This is the boundary where observation ends and scenario construction begins. The source-to-simulation equation is:

$$\text{requested BESS kW} = \text{requested area MW} \times 1000 \times 0.001$$

The `0.001` participation fraction is a fixed research assumption, not an inferred market share. It is applied to every interval and no request is clipped during mapping. For example, 425 MW of area request becomes a 425 kW hypothetical battery request.

The control policy is deliberately simple and inspectable:

1. accept a fixed directional capacity commitment supplied by the study;
2. keep the energy baseline at zero, with no activation lookahead;
3. follow the derived aFRR trajectory or timed mFRR event;
4. let the simulator enforce power, SOC, efficiency, headroom, ramp, response-time and duration constraints; and
5. record any difference between requested and delivered power as non-performance.

This is not a price-threshold controller, bidding optimizer, MPC policy, forecast, or dispatch recommendation. The table exposes upward commitment / downward commitment / accepted baseline / reserved headroom. The aFRR commitment is 450 kW in each direction; the mFRR commitment is 400 kW in each direction. Positive battery power means charging; negative power means discharging to support an upward request.


In [ ]:
battery_assumption = afrr_scenario.payload["battery"]
capability_assumption = afrr_scenario.payload["capability"]
assumption_table = pd.DataFrame([
    {"category": "Battery", "assumption": "Reference asset", "value": f'{battery_assumption["capacity_kwh"]:,.0f} kWh; {battery_assumption["max_charge_kw"]:,.0f}/{battery_assumption["max_discharge_kw"]:,.0f} kW charge/discharge', "evidence_level": "ASSUMED"},
    {"category": "Battery", "assumption": "Initial and allowed SOC", "value": f'{battery_assumption["initial_soc"]:.0%}; {battery_assumption["min_soc"]:.0%}–{battery_assumption["max_soc"]:.0%}', "evidence_level": "ASSUMED"},
    {"category": "Battery", "assumption": "Charge/discharge efficiency", "value": f'{battery_assumption["charge_efficiency"]:.0%}/{battery_assumption["discharge_efficiency"]:.0%}', "evidence_level": "ASSUMED"},
    {"category": "Mapping", "assumption": "Asset allocation", "value": "0.001 × observed area request; no clipping", "evidence_level": "DERIVED"},
    {"category": "Control", "assumption": "Baseline", "value": "0 kW; no activation lookahead", "evidence_level": "ASSUMED"},
    {"category": "Commitment", "assumption": "aFRR / mFRR", "value": "±450 kW / ±400 kW fixed directional capacity", "evidence_level": "ASSUMED"},
    {"category": "Activation", "assumption": "aFRR / mFRR shape", "value": "PT15M interval averages / rectangular 900 s SA event", "evidence_level": "DERIVED"},
    {"category": "Capability", "assumption": "Ramp and maximum response delay", "value": f'{capability_assumption["max_ramp_up_kw_per_second"]:,.0f} kW/s; 1 s', "evidence_level": "ASSUMED"},
    {"category": "Economics", "assumption": "Offline fixed inputs", "value": "RTE activation-price proxy; capacity/day-ahead = €0; penalty unconfigured; degradation = €0.02/kWh throughput", "evidence_level": "ASSUMED / PROXY"},
])
display(assumption_table)

commitment_table = pd.DataFrame(display_payload["commitment_rows"])
commitment_table["reserved_power_headroom_kw"] = commitment_table["committed_capacity_kw"]
activation_table = pd.DataFrame(display_payload["activation_rows"])
display(commitment_table)
display(activation_table)


## 5. Replay the derived request through the battery

The next comparison is the heart of the demonstration. **Derived requested power** is the scenario input after applying the allocation assumption. **Simulated delivered power** is what Open Energy Lab's battery model supplies after enforcing the declared physical and service constraints. The zero line is the accepted baseline.

The shaded band represents the fixed capacity commitment. A request outside that band would violate the service commitment before battery physics is considered. A gap between the request and delivery lines indicates constrained delivery and should be explained by the performance audit—not hidden or imputed.


In [ ]:
physical_table = pd.DataFrame(display_payload["physical_rows"])
power_plot = physical_table.copy()
power_plot["start_utc"] = pd.to_datetime(power_plot["start_utc"], utc=True)
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)
for axis, product in zip(axes, ["AFRR", "MFRR"], strict=True):
    group = power_plot[power_plot["product"] == product].sort_values("start_utc")
    capacity_kw = commitment_table[commitment_table["product"] == product]["committed_capacity_kw"].max()
    axis.fill_between(group["start_utc"], -capacity_kw, capacity_kw, color="#D9D9D9", alpha=0.45, label=f"fixed ±{capacity_kw:.0f} kW commitment")
    axis.plot(group["start_utc"], group["requested_balancing_power_kw"], marker="o", linewidth=2.8, label="DERIVED — requested balancing")
    axis.plot(group["start_utc"], group["delivered_balancing_power_kw"], marker="x", linewidth=2, linestyle="--", label="SIMULATED — delivered balancing")
    axis.plot(group["start_utc"], group["accepted_baseline_power_kw"], color="black", linestyle=":", label="ASSUMED — zero baseline")
    axis.set_title(f"{product}: scenario request versus constrained battery response")
    axis.set_ylabel("Battery power (kW; charge positive)")
    axis.legend(fontsize=8, ncol=2)
axes[-1].set_xlabel("UTC interval start")
fig.tight_layout()
generated_figures["requested_delivered_power"] = fig
plt.show()


### 5.1 Did the battery deliver what the scenario requested?

If the requested and delivered lines overlap, that is a result: the reference battery met the interval-average request under these assumptions. The detailed tables make the apparent overlap auditable through requested balancing power / delivered balancing power / combined power and tracking error / response delay / achieved ramp / non-performance / causes.

This result should be read narrowly. For aFRR, exact PT15M delivery does not prove that a battery could follow an unavailable sub-quarter-hour AGC signal. For mFRR, it validates the declared rectangular SA event, not the shape of an actual asset order.


In [ ]:
performance_table = pd.DataFrame(display_payload["performance_rows"])
display(physical_table)
display(performance_table)
# requested balancing power / delivered balancing power / combined power
# tracking error / response delay / achieved ramp / non-performance / causes


### 5.2 SOC and headroom explain feasibility

**SOC** is the fraction of usable energy stored in the battery. **Headroom** is the response still available after accounting for the current baseline, battery limits and reserved service capacity:

- upward power/energy headroom indicates room to discharge more;
- downward power/energy headroom indicates room to charge more.

Power headroom answers “how many additional kW could be delivered now?” Energy headroom answers “how many kWh remain before an SOC boundary?” The cases use different dates, so the graphic gives each product its own time axis rather than drawing a misleading line across the gap.


In [ ]:
headroom_plot = physical_table.copy()
headroom_plot["start_utc"] = pd.to_datetime(headroom_plot["start_utc"], utc=True)
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex="col")
for column, product in enumerate(["AFRR", "MFRR"]):
    group = headroom_plot[headroom_plot["product"] == product].sort_values("start_utc")
    axes[0, column].plot(group["start_utc"], group["soc"], marker="o", color="#4C78A8")
    axes[0, column].axhspan(battery_assumption["min_soc"], battery_assumption["max_soc"], alpha=0.12, color="#54A24B", label="allowed SOC band")
    axes[1, column].plot(group["start_utc"], group["upward_power_headroom_kw"], marker="o", label="upward (discharge)")
    axes[1, column].plot(group["start_utc"], group["downward_power_headroom_kw"], marker="x", label="downward (charge)")
    axes[2, column].plot(group["start_utc"], group["upward_energy_headroom_kwh"], marker="o", label="upward (discharge)")
    axes[2, column].plot(group["start_utc"], group["downward_energy_headroom_kwh"], marker="x", label="downward (charge)")
    axes[0, column].set_title(f"SIMULATED — {product}")
    axes[2, column].set_xlabel("UTC interval start")
    for row in range(3):
        axes[row, column].legend(fontsize=8)
axes[0, 0].set_ylabel("SOC (fraction)")
axes[1, 0].set_ylabel("Power headroom (kW)")
axes[2, 0].set_ylabel("Energy headroom (kWh)")
fig.suptitle("Physical feasibility: SOC / upward power headroom / downward power headroom / energy headroom", y=1.01)
fig.tight_layout()
generated_figures["soc_headroom"] = fig
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
tracking_summary = physical_table.groupby("product", as_index=False)[["absolute_tracking_error_kw", "non_performance_duration_seconds"]].sum()
sns.barplot(data=tracking_summary, x="product", y="absolute_tracking_error_kw", ax=axes[0], color="#E45756")
sns.barplot(data=tracking_summary, x="product", y="non_performance_duration_seconds", ax=axes[1], color="#B279A2")
axes[0].set_title("SIMULATED — total absolute tracking error")
axes[1].set_title("SIMULATED — non-performance duration")
for axis in axes:
    for container in axis.containers:
        axis.bar_label(container, fmt="%.2f", padding=3)
    if axis.get_ylim()[1] == 0:
        axis.set_ylim(0, 1)
fig.tight_layout()
generated_figures["tracking_performance"] = fig
plt.show()


### 5.3 Energy and throughput translate power into battery use

Power is instantaneous; energy accumulates it over time. Charge energy is energy entering the battery terminal, discharge energy is energy leaving it, net terminal energy is their signed balance, and throughput is the total absolute energy moved. Throughput is used only for the declared degradation-cost proxy.

**What to look for:** the upward aFRR example predominantly discharges, while the downward mFRR example charges. This directional behavior follows the derived requests; it is not chosen in response to a price threshold.


In [ ]:
energy_table = pd.DataFrame(display_payload["energy_rows"])
display(energy_table)
# charge / discharge / net terminal energy / throughput


In [ ]:
energy_labels = {"charge_energy_kwh": "Charge", "discharge_energy_kwh": "Discharge", "net_terminal_energy_kwh": "Net terminal", "throughput_kwh": "Throughput"}
energy_long = energy_table.melt(id_vars="product", value_vars=list(energy_labels), var_name="metric", value_name="kWh")
energy_long["metric"] = energy_long["metric"].map(energy_labels)
fig, axis = plt.subplots(figsize=(11, 5))
sns.barplot(data=energy_long, x="metric", y="kWh", hue="product", ax=axis)
axis.set_title("SIMULATED — battery-terminal energy and throughput")
axis.tick_params(axis="x", rotation=20)
fig.tight_layout()
generated_figures["energy_throughput"] = fig
plt.show()


## 6. Reconcile a transparent scenario settlement

Settlement is shown component by component so assumptions cannot disappear inside one revenue number:

- **day-ahead energy:** zero here because the accepted energy baseline and day-ahead price assumption are zero; ENTSO-E is not used in the fixed offline calculation;
- **capacity:** zero because no validated capacity-price input is supplied; the physical commitment still exists;
- **activated energy:** uses the relevant RTE area price as a `scenario_proxy`;
- **non-performance:** reported physically, but the monetary penalty is unconfigured and therefore zero; and
- **degradation:** throughput multiplied by €0.02/kWh, a comparison proxy rather than an electrochemical aging model.

Gross revenue adds market-revenue components. Net result then subtracts configured penalty and degradation costs. These values demonstrate deterministic accounting, not an asset invoice or realized profitability.


In [ ]:
settlement_table = pd.DataFrame(display_payload["settlement_rows"])
display(settlement_table)
# day-ahead / capacity / activated energy / penalty / degradation / gross / net


In [ ]:
settlement_plot = settlement_table.copy()
settlement_plot["capacity_revenue_eur"] = settlement_plot["upward_capacity_revenue_eur"] + settlement_plot["downward_capacity_revenue_eur"]
settlement_plot["activation_revenue_eur"] = settlement_plot["upward_activation_revenue_eur"] + settlement_plot["downward_activation_revenue_eur"]
settlement_plot["penalty_cost_eur"] = -settlement_plot["non_performance_penalty_eur"]
settlement_plot["degradation_cost_signed_eur"] = -settlement_plot["degradation_cost_eur"]
settlement_labels = {"day_ahead_energy_revenue_eur": "Day-ahead", "capacity_revenue_eur": "Capacity", "activation_revenue_eur": "Activated energy", "penalty_cost_eur": "Penalty", "degradation_cost_signed_eur": "Degradation"}
settlement_long = settlement_plot.melt(id_vars="product", value_vars=list(settlement_labels), var_name="component", value_name="EUR")
settlement_long["component"] = settlement_long["component"].map(settlement_labels)
result_long = settlement_plot.melt(id_vars="product", value_vars=["gross_revenue_eur", "net_result_eur"], var_name="result", value_name="EUR")
result_long["result"] = result_long["result"].map({"gross_revenue_eur": "Gross revenue", "net_result_eur": "Net result"})
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2, 1]})
sns.barplot(data=settlement_long, x="component", y="EUR", hue="product", ax=axes[0])
sns.barplot(data=result_long, x="result", y="EUR", hue="product", ax=axes[1])
axes[0].axhline(0, color="black", linewidth=0.8)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("ASSUMED / PROXY — settlement components")
axes[1].set_title("CALCULATED — gross versus net")
axes[0].tick_params(axis="x", rotation=20)
fig.tight_layout()
generated_figures["settlement"] = fig
plt.show()


## 7. Verify reproducibility and lineage

The final offline audit records artifact / mapping / commitment / activation / replay / settlement / display fingerprints. A fingerprint changes when the corresponding semantic input changes. This lets a researcher distinguish “the same source with the same assumptions” from a revised source, mapping, commitment, model result, or display.

The notebook rebuilt the recorded result bytes near the start and asserted equality. The table below exposes the identities used in that check. Figure export remains explicitly disabled because the executed visuals contain restricted data.


In [ ]:
reproducibility_table = pd.DataFrame(display_payload["reproducibility_rows"])
assert notebook_display_fingerprint == rebuilt_results["notebook_display_fingerprint"]
display(reproducibility_table)
# artifact / mapping / commitment / activation / replay / settlement / display fingerprints


In [ ]:
EXPORT_FIGURES = False
FIGURE_OUTPUT_DIRECTORY = REPO_ROOT / "tmp/restoration-services-notebook/figures"
if EXPORT_FIGURES:
    FIGURE_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    for figure_name, figure in generated_figures.items():
        figure.savefig(FIGURE_OUTPUT_DIRECTORY / f"{figure_name}.png", dpi=150, bbox_inches="tight")
else:
    print("Figure export disabled; committed notebook state stays output-free.")


## 8. Optional live acquisition and composition check

This section is deliberately outside the demonstration's offline evidence chain. The next cell is disabled by default and accepts a caller-selected UTC half-open period. When `RUN_LIVE_QUERY=True`, it runs the **whole live pipeline**: in-memory RTE Balancing Energy 5.2 acquisition with short-lived OAuth, normalized artifact construction, directional mapping, fixed commitments, zero baseline, BESS replay, delivery validation, degradation proxy and settlement. Raw responses remain in memory only for RTE and no raw response is written. Live availability is not required for offline results.

| Guard | Purpose | Default |
|---|---|---|
| `RUN_LIVE_QUERY` | Fetch RTE data and run it through mapping, replay and settlement | `False` |
| `RUN_LIVE_ENTSOE_QUERY` | Acquire native PT15M A44 day-ahead prices in memory and use them in settlement | `False` |
| `LIVE_ENTSOE_ARTIFACT_PATH` | Alternative: compose an explicitly supplied compatible ENTSO-E artifact | `None` |
| `LIVE_PERSIST_NORMALIZED_ARTIFACT` | Persistence is rejected in this notebook; use the governed CLI instead | `False` |

ENTSO-E remains optional and independent. If neither ENTSO-E option is selected, the pipeline explicitly uses the declared zero day-ahead assumption. The direct live branch uses the additive `fr-day-ahead-quarter-hour-v2` profile: it requests A44 only, requires native `PT15M`, preserves the four quarter-hours in each hour without interpolation, and builds a source-separated artifact in memory. This matches RTE's PT15M intervals exactly. Existing hourly Feature 010 artifacts remain valid for historical composition and are not changed. The A44 price affects only day-ahead baseline settlement; it is never used as an aFRR/mFRR activation command or automatic fallback.

`LIVE_SCENARIO_PARAMETERS` makes the directional allocation and commitment explicit for each RTE resource. A source-null driver, an allocation above commitment, missing prices, or incompatible ENTSO-E coverage fails with diagnostics rather than being imputed.


In [ ]:
RUN_LIVE_QUERY = True
RUN_LIVE_ENTSOE_QUERY = True
LIVE_PERSIST_NORMALIZED_ARTIFACT = False
LIVE_QUERY_PARAMETERS = {
    "resource": "standard_afrr_data",
    "requested_start_utc": "2026-08-20T00:00:00Z",
    "requested_end_utc": "2026-08-20T01:00:00Z",
}
LIVE_SCENARIO_PARAMETERS = {
    "standard_afrr_data": {"direction": "UPWARD", "asset_allocation_fraction": 0.001, "committed_capacity_kw": 450.0},
    "standard_mfrr_data": {"direction": "DOWNWARD", "asset_allocation_fraction": 0.001, "committed_capacity_kw": 400.0},
}
LIVE_ENV_FILE = REPO_ROOT / ".env"
LIVE_ENTSOE_ARTIFACT_PATH = None
live_rte_table = pd.DataFrame()
live_replay_table = pd.DataFrame()
live_settlement_table = pd.DataFrame()
live_pipeline_result = None
live_entsoe_selection = "not requested; using declared zero day-ahead price"
raw_responses = ()

if RUN_LIVE_QUERY:
    from examples.restoration_services.fetch_rte_period import _rte_credentials
    from examples.restoration_services.live_pipeline import run_live_restoration_pipeline
    from open_energy_lab.market.artifact import load_market_data_artifact
    from open_energy_lab.market.rte import (
        RTE_PROFILES,
        build_rte_restoration_artifact_from_responses,
        build_rte_restoration_request,
        execute_rte_request,
    )

    if LIVE_PERSIST_NORMALIZED_ARTIFACT:
        raise ValueError("the notebook live pipeline does not persist artifacts; use the explicit fetch CLI")
    if RUN_LIVE_ENTSOE_QUERY and LIVE_ENTSOE_ARTIFACT_PATH is not None:
        raise ValueError("choose either RUN_LIVE_ENTSOE_QUERY or LIVE_ENTSOE_ARTIFACT_PATH, not both")
    profile = RTE_PROFILES[LIVE_QUERY_PARAMETERS["resource"]]
    live_start = pd.Timestamp(LIVE_QUERY_PARAMETERS["requested_start_utc"]).to_pydatetime()
    live_end = pd.Timestamp(LIVE_QUERY_PARAMETERS["requested_end_utc"]).to_pydatetime()
    request = build_rte_restoration_request(
        resource=profile.resource,
        requested_start_utc=live_start,
        requested_end_utc=live_end,
        requested_series_ids=profile.series_ids,
        required_series_ids=profile.required_series_ids,
        normalization_version="rte-balancing-energy-v5.2-notebook-live-v1",
        raw_retention_policy="DIGEST_ONLY",
        mode="FETCH",
    )
    client_id = ""
    client_secret = ""
    api_token = ""
    try:
        client_id, client_secret = _rte_credentials(LIVE_ENV_FILE)
        raw_responses = execute_rte_request(request, client_id=client_id, client_secret=client_secret)
        live_rte_artifact = build_rte_restoration_artifact_from_responses(request, raw_responses)
        raw_responses = ()
        live_rows = []
        for series in live_rte_artifact.series:
            for observation in series.observations:
                live_rows.append({
                    "series_id": series.series_id,
                    "source_field": series.source_field,
                    "direction": series.direction,
                    "unit": series.canonical_unit,
                    "resolution": series.canonical_resolution,
                    "start_utc": observation.interval.start_utc,
                    "value_state": observation.value_state.value,
                    "value": observation.value,
                    "evidence_level": "OBSERVED_AREA_AGGREGATE",
                })
        live_rte_table = pd.DataFrame(live_rows)
        display(live_rte_table)

        entsoe_artifact = None
        if RUN_LIVE_ENTSOE_QUERY:
            from examples.bess_stacked_services.fetch_french_period import _api_token
            from open_energy_lab.market.entsoe import DEFAULT_ENTSOE_ENDPOINT, EntsoeMarketDataRequest, FranceQuarterHourDayAheadProfile
            from open_energy_lab.market.entsoe_restoration import fetch_entsoe_quarter_hour_day_ahead

            entsoe_request = EntsoeMarketDataRequest(
                requested_start_utc=live_start,
                requested_end_utc=live_end,
                source_endpoint=DEFAULT_ENTSOE_ENDPOINT,
                raw_retention_policy="digest_only",
                normalization_version="entsoe-fr-a44-native-pt15m-v2",
                profile=FranceQuarterHourDayAheadProfile(),
            )
            api_token = _api_token(LIVE_ENV_FILE)
            entsoe_fetch = fetch_entsoe_quarter_hour_day_ahead(entsoe_request, api_token=api_token)
            entsoe_artifact = entsoe_fetch.artifact
            api_token = ""
            live_entsoe_selection = {"mode": "in_memory_live_query", "profile_id": entsoe_artifact.profile_id, "resolution": "PT15M", "artifact_fingerprint": entsoe_artifact.artifact_fingerprint, "series_id": "day_ahead_price"}
        elif LIVE_ENTSOE_ARTIFACT_PATH is not None:
            entsoe_artifact = load_market_data_artifact(Path(LIVE_ENTSOE_ARTIFACT_PATH))
            live_entsoe_selection = {"mode": "existing_artifact", "profile_id": entsoe_artifact.profile_id, "artifact_fingerprint": entsoe_artifact.artifact_fingerprint, "series_id": "day_ahead_price"}

        live_pipeline_result = run_live_restoration_pipeline(
            live_rte_artifact,
            entsoe_day_ahead_artifact=entsoe_artifact,
            **LIVE_SCENARIO_PARAMETERS[profile.resource],
        )
        if entsoe_artifact is None:
            live_entsoe_selection = live_pipeline_result.day_ahead_source
        live_replay_table = pd.DataFrame([
            {
                "start_utc": sample.interval.start_utc,
                "requested_balancing_power_kw": sample.requested_balancing_power_kw,
                "delivered_balancing_power_kw": sample.delivered_balancing_power_kw,
                "soc": sample.state.soc,
                "upward_power_headroom_kw": sample.upward_power_headroom_kw,
                "downward_power_headroom_kw": sample.downward_power_headroom_kw,
                "tracking_error_kw": sample.tracking_error_kw,
                "causes": ", ".join(sample.causes) or "none",
                "evidence_level": "SIMULATED",
            }
            for sample in live_pipeline_result.replay.samples
        ])
        settlement_payload = live_pipeline_result.settlement.to_dict()
        live_settlement_table = pd.DataFrame([{
            "availability": settlement_payload["availability"],
            "day_ahead_source": live_pipeline_result.day_ahead_source,
            "day_ahead_energy_revenue_eur": settlement_payload.get("day_ahead_energy_revenue_eur"),
            "capacity_revenue_eur": None if settlement_payload.get("upward_capacity_revenue_eur") is None else settlement_payload["upward_capacity_revenue_eur"] + settlement_payload["downward_capacity_revenue_eur"],
            "activation_revenue_eur": None if settlement_payload.get("upward_activation_revenue_eur") is None else settlement_payload["upward_activation_revenue_eur"] + settlement_payload["downward_activation_revenue_eur"],
            "non_performance_penalty_eur": settlement_payload.get("non_performance_penalty_eur"),
            "degradation_cost_eur": settlement_payload.get("degradation_cost_eur"),
            "net_result_eur": settlement_payload.get("net_result_eur"),
            "diagnostics": settlement_payload["unavailable_reasons"],
            "evidence_level": "CALCULATED_SCENARIO",
        }])
        display(live_replay_table)
        display(live_settlement_table)
        display(pd.DataFrame([live_entsoe_selection]) if isinstance(live_entsoe_selection, dict) else pd.DataFrame([{"day_ahead_source": live_entsoe_selection}]))

        fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
        axes[0].plot(live_replay_table["start_utc"], live_replay_table["requested_balancing_power_kw"], marker="o", label="DERIVED request")
        axes[0].plot(live_replay_table["start_utc"], live_replay_table["delivered_balancing_power_kw"], marker="x", linestyle="--", label="SIMULATED delivery")
        axes[0].set_ylabel("Power (kW)")
        axes[0].legend()
        axes[1].plot(live_replay_table["start_utc"], live_replay_table["soc"], marker="o", color="#4C78A8")
        axes[1].set_ylabel("SOC")
        axes[2].plot(live_replay_table["start_utc"], live_replay_table["upward_power_headroom_kw"], marker="o", label="upward")
        axes[2].plot(live_replay_table["start_utc"], live_replay_table["downward_power_headroom_kw"], marker="x", label="downward")
        axes[2].set_ylabel("Headroom (kW)")
        axes[2].set_xlabel("UTC interval start")
        axes[2].legend()
        fig.suptitle(f"LIVE {live_pipeline_result.product.value} — observed source to simulated BESS result")
        fig.tight_layout()
        plt.show()
    finally:
        client_id = ""
        client_secret = ""
        api_token = ""
        raw_responses = ()
else:
    print("Live pipeline disabled; set RUN_LIVE_QUERY=True only for an explicit source-to-settlement run.")


## 9. Conclusions: what the demonstration establishes

> **Takeaway:** Open Energy Lab can turn a transparent, explicitly assumed restoration scenario into a deterministic and auditable BESS replay. It does not turn public area aggregates into an observed battery control signal.

| We demonstrated | Evidence level | We did **not** demonstrate |
|---|---|---|
| Exact parsing and provenance for distinct RTE aFRR and mFRR series | **OBSERVED** market-area evidence | An asset-level automatic or manual dispatch command |
| Fixed mapping, commitments, zero baseline and event shape | **ASSUMED / DERIVED** scenario | Bid selection, optimal scheduling, forecasting, or market award |
| Battery power/SOC/efficiency/service limits, delivery and headroom | **SIMULATED** result | PT15M proof of second-by-second aFRR tracking |
| Explicit activation-price proxy, degradation proxy and net arithmetic | **CALCULATED** scenario result | A contract, invoice, penalty regime, or profitability guarantee |

The example proves that the library can preserve exact RTE source semantics, build separate aFRR and mFRR inputs, reserve fixed directional capacity, apply BESS power/SOC/efficiency/service limits, audit delivery and headroom, and reconcile an explicit scenario-proxy settlement deterministically.

It does not prove that the public area aggregate was sent to this battery, that PT15M aFRR captures operational tracking, that the rectangular SA event was the actual order shape, that a DA event occurred, or that the proxy revenues would be realized. Operational control signals, asset contracts, bid selection, imbalance rules, prequalification, and forecast risk require additional sources and later features.
